In [1]:
# goal: prototype skeleton transformer
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from map4 import MAP4
import re

import matplotlib.pyplot as plt
%matplotlib inline

from pathlib import Path

In [2]:
import numpy as np
import torch

class RadialSeeker:

    def __init__(self, radial_resolution, intrashell_resolution, max_angstroms,
                 verbose=False):
        self.radial_resolution = radial_resolution
        self.angstrom_lim = max_angstroms  # maximum angstrom range found + some extra for context enrichment
        # can always edit this later on
        self.angstrom_inc = float(max_angstroms / radial_resolution)
        self.threshold = self.angstrom_inc / 2  # standardize how we determine radial sequences
        # getting half the angstrom increment will let us finely separate them

        #                           smallest distance is the base increment, not 0
        self.radius_levels = torch.arange(self.angstrom_lim, self.angstrom_inc, step=-1 * self.angstrom_inc)

        self.intrashell_resolution = intrashell_resolution
        self.intrashell_inc = max_angstroms / intrashell_resolution

        self.verbose = verbose

    def radial_sequence(self, aa_seq, vect_seq):
        # first order them by absolute distance to centroid
        # then convert them to indices according to intra-shell resolution

        # radial resolution determines how finely we want to order them
        # shell resolution determines how finely in 3D space we represent them

        radial_seq = []  # lookup table
        seen = set()

        # sanity
        if len(aa_seq) != len(vect_seq):
            raise ValueError(f"string and vector sequences of {aa_seq} are different lengths")

        # iterate up resolution increments, if a molecule's coordinate is within like 1/resolution of an angstrom, append it's info
        # its (radius, pos index) is now the unique ID, so if we stumble on it again its not duplicated
        for level in self.radius_levels:  # go through all possible radius levels
            for i in range(len(aa_seq)):  # go through all amino acids in that sequence
                num_id = i
                if num_id not in seen and self._dist_check(np.linalg.norm(vect_seq[i]), level):
                    idx_vect = self.vect2idx(vect_seq[i])
                    radial_seq.append([[aa_seq[i]], idx_vect])
                    seen.add(num_id)

        # create two VOID tokens (0,0,0) on both sides to denote stops and starts
        return radial_seq + [[['VOID'], [0, 0, 0]]]  # we want to go from outside inward

    def _dist_check(self, dist, ang_radius):
        if abs(dist - ang_radius) <= self.threshold:
            return True
        else:
            return False

    def vect2idx(self, vector):
        """
        ENCODE: Converts a tensor of 3D vectors into their nearest shell-resolution's index
        """
        vector_a = np.array(vector) + self.angstrom_lim
        idxs = np.round(vector_a / (2 * self.intrashell_inc))
        idxs = np.clip(idxs, 0, self.intrashell_resolution)
        return idxs.astype(int).tolist()

    def num2vect(self, idxs):
        """DECODE: Converts a tensor of indexes into their max_Angstrom-scaled 3D vectors"""
        vector = []
        for number in idxs:
            vector.append(float(number) * 2 * self.intrashell_inc - self.angstrom_lim)
        return vector


def test_resolution():
    module = RadialSeeker(radial_resolution=100, intrashell_resolution=100, max_angstroms=33)
    for level in module.radius_levels:
        print(level)

    test_vector = [0.6, 1.2, 23.69]
    t_v2ix = module.vect2idx(test_vector)
    t_ix4v = module.num2vect(t_v2ix)
    print(f"{test_vector} --Encode-- {t_v2ix}")
    print(f"{t_v2ix} --Decode-- {t_ix4v}")
# test_resolution()  # all good

In [3]:
mol_dim = 1024
map4_fprinter = MAP4(
    dimensions=mol_dim,
    radius=2,
    include_duplicated_shingles=True,
)

# will be important later for visualization of skeleton
def eval_lipinski(smiles):
    """
    Use on user-input smiles, don't shut down inference run just flag molecule as non drug-like
    :param smiles:
    :return:
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES String")

    try:
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        h_donors = Lipinski.NumHDonors(mol)
        h_acceptors = Lipinski.NumHAcceptors(mol)
    except Exception as e:
        raise ValueError(f"Error calculating Lipinski Descriptors: {e}")

    rules_passed = 0
    if mw <= 500: rules_passed += 1
    if logp <= 5: rules_passed += 1
    if h_donors <= 5: rules_passed += 1
    if h_acceptors <= 10: rules_passed += 1

    return rules_passed

def mol_from_smiles(smiles):
    """
    Extract molecular fingerprint from singular SMILES - fairly straightforward
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        map4_fp = map4_fprinter.calculate(mol=mol)

        return map4_fp

    except Exception as e:
        return None

In [4]:
# Load data
# print(SMILE_2_PDBHITS.columns)  # 'SMILES', 'PDB_hits'
# print(MOLECULAR_FINGERPRINTS.columns)  # 'smiles_str', 'map4_fp'
# print(RADIAL_SEQUENCES.columns)  # 'PDB_ID', 'radial_sequence'
data_folder = Path("notebook_database")
RADIAL_SEQUENCES = pd.read_pickle(data_folder/"radial_seq_df.pkl")
MOLECULAR_FINGERPRINTS = pd.read_pickle(data_folder/"molfp_df.pkl")
TRAIN_POINTER = pd.read_parquet(data_folder/"training_pointers.parquet")
TEST_POINTER = pd.read_parquet(data_folder/"test_pointers.parquet")

In [ ]:
import torch
from torch.utils.data import Dataset

class RadialDataset(Dataset):

    def __init__(self, test_pointer, train_pointer, smiles_molfp, pdb_radial,
                 block_size, mode='train'):
        self.test_pointer = test_pointer
        self.train_pointer = train_pointer                             # when extracted:
        self.smiles_molfp = smiles_molfp.set_index("smiles_str")       # <class 'torch.Tensor'>
        self.pdb_radial = pdb_radial.set_index("PDB_ID")               # <class 'list'>
        # set index moves "column label" column to row label, enabling O(1) .loc["target_ID"] lookups

        self.block_size = block_size
        self.mode = mode if mode =='train' else 'test'


    def __len__(self):
        return len(self.train_pointer) if self.mode == 'train' else len(self.test_pointer)


    def __getitem__(self, idx):
        row = self.train_pointer.iloc[idx] if self.mode == 'train' else self.test_pointer.iloc[idx]
        smiles = row["SMILES"]
        pdb_id = row["PDB_HIT"]
        target_idx = row["WINDOW_IDX"]

        # context query
        molecular_fingerprint = self.smiles_molfp.loc[smiles, "map4_fp"]

        # radial sequence query
        radial_sequence = self.pdb_radial.loc[pdb_id, "radial_sequence"]

        radial_X, radial_Y = self.radial_XY(radial_sequence, target_idx)

        return (molecular_fingerprint,
                torch.tensor(radial_X, dtype=torch.float32),     # block_size, 3
                torch.tensor(radial_Y, dtype=torch.float32))     # 3,


    def radial_XY(self, radial_seq, target_idx):
        """
        Generates X and Y set for a given radial sequence + molecular fingerprint in the background?
        """
        context = [[0, 0, 0] for _ in range(self.block_size)]
        for idx in range(len(radial_seq)):
            coords = radial_seq[idx][1]  # = [X, Y, Z]
            if idx == target_idx:
                return context, coords  # This is our XY data
            context = context[1:]+[coords]  # sliding window append -> context style in nanogpt

        raise ValueError(f"target idx out of range for {radial_seq}")

In [6]:
# RadialSeeker parameters used -> record these - might design a config file later
radial_resolution = 100
intrashell_resolution = 100
max_angstroms = 33

batch_size = 360  # how many data X X Ys we want to pass at a time
block_size = 10  # coordinate context
max_iters = 500
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 128
n_head = 4
n_layer = 3
dropout = 0.2
coord_num = 101

# Dataset parameters
training_dataset = RadialDataset(test_pointer=TEST_POINTER, train_pointer=TRAIN_POINTER,
                        smiles_molfp=MOLECULAR_FINGERPRINTS, pdb_radial=RADIAL_SEQUENCES,
                        block_size=block_size,
                                 mode='train')

# DataLoader
from torch.utils.data import DataLoader
loader = DataLoader(
    training_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True
)

torch.manual_seed(311104)

In [117]:
class SkeletonMaker(nn.Module):
    def __init__(self, c_context, mol_fingerprint):
        super().__init__()
        self.coordinates_in = nn.Linear(3, n_embd)
        self.position_embedding = nn.Embedding(c_context, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)

        self.coordinates_out = nn.Linear(n_embd, 3)


    def forward(self, idx, targets=None):
        B, T, _ = idx.shape  # B, T, C -> triplets
        coord_emb = self.coordinates_in(idx)
        position_emb = self.position_embedding(torch.arange(T))
        x = coord_emb + position_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.coordinates_out(x)

        if targets is None:
            loss = None
        else:
            last_pred = logits[:, -1, :]
            loss = F.mse_loss(last_pred, targets.float())

        return logits, loss


    def generate(self, ligand_fp, coord_context):
        coordinates = []
        ligand_fp = None   # will set molecular fingerprint in here
        while True:
            idx_cond = coord_context
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            coordinate_sample = torch.multinomial(probs, num_samples=1)
            print(f"Coordinate_sample: {coordinate_sample}")
            coordinates.append(coordinate_sample)
            if 0 in coordinate_sample:
                break
        return coordinates

tensor([[[ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.]],

        [[ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.]],

        [[ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [35., 38., 37.]],

        [[ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
         [ 0.,  0.,  0.],
      

In [72]:
# data loading
class Head(nn.Module):
    """one head of self-attention"""
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)     # linear projections
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # this is just what you have to do to create the lower triangular matrix thing
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)   # both (B, T, C)

        # compute attention scores ("affinities")
        attn_wts = q @ k.transpose(-2, -1) * C**-0.5  # (B, T, C) @ (B, C, T) -> (B, T, T)
        attn_wts = attn_wts.masked_fill(self.tril[:T, :T] == 0, float('-inf'))   # (B, T, T)
        attn_wts = F.softmax(attn_wts, dim=-1)
        attn_wts = self.dropout(attn_wts)

        # weighted aggregations
        v = self.value(x)
        out = attn_wts @ v  # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out


class MultiHeadAttention(nn.Module):  # later will change this to MH-cross-attention
    """multiple heads of self-attention in parallel"""
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        # introduce a projection
        # run all of these single head self attention heads in parallel into a list
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)  # output of self.attention itself + applying a linear transformation of the outcome
        # concatenate all of the outputs
        out = self.dropout(out)
        return out


class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),   # projection layer going back into the residual pathway
            nn.Dropout(dropout)
        )
        # multiplied n_embd by 4 because that's what the paper did
        # adding more computation and growing the layer in the residual block on the side

    def forward(self, x):
        return self.net(x)


class LayerNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        # backprop-trained parameters
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x):
        # calculate forward pass
        xmean = x.mean(0, keepdim=True)
        xvar = x.var(0, keepdim=True)
        xhat = (x-xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta

        return self.out

    def parameters(self):
        return [self.gamma, self.beta]


class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.satt_head = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # applied layer norm pre-formulation
        x = x + self.satt_head(self.ln1(x))  # residual connections
        x = x + self.ffwd(self.ln2(x))
        # for these two, we're basically forking off, doing some computation and returning
        return x